# Simple TabM Baseline for Playground Series S6E9

A minimal, single-model baseline for predicting `Will_Buy_EV` with **TabM**
(Gorishniy et al., 2024: *TabM: Advancing Tabular Deep Learning with Parameter-Efficient Ensembling*)
through the `pytabkit` implementation.

**What this notebook does**

- Uses only the competition data (no external datasets, no target encoding).
- Builds a few target-free features: raw columns, decimal digits of the numeric columns, and the
  exact-value frequency of `Annual_Income_USD` over train + test.
- Trains one `TabM_D_Classifier` per fold with stratified 5-fold CV, reports the out-of-fold ROC AUC,
  and averages the 5 fold models on the test set for `submission.csv`.

**Why TabM here**

Gradient-boosted trees dominate this competition, but in my local experiments the models that added
the most to a large stacked ensemble were neural networks that *do not* see target encodings.
A plain TabM on the raw columns plus digit features reaches roughly 0.9447 OOF AUC, and it is far less
correlated with the tree models than any other single model I tried. This notebook is that model,
kept as simple as possible so it is easy to fork.

Set `SMOKE = True` to run a quick end-to-end check on a subsample before a full GPU run, and
`FAST = False` if you want the (slower, slightly stronger) pytabkit default hyperparameters.


In [ ]:
import subprocess, sys, torch
# Keep Kaggle's preinstalled torch: a newer CUDA build from pip may not include kernels for the notebook GPU.
open("constraints.txt", "w").write(f"torch=={torch.__version__}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "constraints.txt", "pytabkit==1.7.3"], check=True)
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from pytabkit import TabM_D_Classifier

# pytabkit 1.7.3 still calls scikit-learn with the old `force_all_finite` argument name,
# which only raises a FutureWarning; silence that one message so the fold log stays readable.
warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")

SMOKE = False            # True: subsample + 2 epochs, just to check the pipeline
FAST = True              # True: small TabM ensemble + large batches (minutes per fold on a T4)
SEED = 42
N_SPLITS = 5
def pick_device() -> str:
    if not torch.cuda.is_available():
        return "cpu"
    try:  # make sure the CUDA build actually runs on this GPU
        (torch.ones(8, device="cuda") @ torch.ones(8, device="cuda")).item()
        return "cuda"
    except Exception as error:  # noqa: BLE001
        print("CUDA unusable, falling back to CPU:", error)
        return "cpu"


DEVICE = pick_device()
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| cuda", torch.version.cuda)
import glob
_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
DATA_DIR = os.path.dirname(_candidates[0]) if _candidates else "/kaggle/input/playground-series-s6e9"
print("data dir:", DATA_DIR)
print("device:", DEVICE)


In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
TARGET = "Will_Buy_EV"
ID = "id"

y = train[TARGET].map({"No": 0, "Yes": 1}).fillna(train[TARGET]).astype(int).to_numpy()
print(train.shape, test.shape, "positive rate:", y.mean().round(4))


## Features

Everything below is target-free, so it can be computed once on train + test without leakage.

- **Raw columns**: 7 numeric, 4 categorical (kept as strings for TabM's categorical embeddings),
  2 yes/no flags as 0/1.
- **Digits**: each decimal digit of the numeric columns as a categorical token. The synthetic
  generator reuses exact values, and digit tokens let the network pick that up cheaply.
- **Income frequency**: how many rows (train + test) share the exact `Annual_Income_USD` value.


In [ ]:
NUMERIC = [
    "Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level",
]
CATEGORICAL = ["Gender", "City_Type", "Current_Car_Type", "Range_Anxiety_Level"]
FLAGS = ["Home_Charging_Possible", "Subsidy_Available"]
DIGIT_SPECS = {  # column: (scale, number of digits)
    "Age": (1.0, 2), "Annual_Income_USD": (1.0, 6), "Daily_Commute_km": (10.0, 4),
    "Number_of_Cars_Owned": (1.0, 1), "Charging_Stations_Near_Home": (1.0, 2),
    "Charging_Stations_Near_Work": (1.0, 2), "Environmental_Concern_Level": (1.0, 1),
}


def build_features(frame: pd.DataFrame, income_counts: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=frame.index)
    for c in NUMERIC:
        out[c] = pd.to_numeric(frame[c], errors="coerce").astype(float)
    for c in CATEGORICAL:
        out[c] = frame[c].fillna("__MISSING__").astype(str)
    for c in FLAGS:
        out[c] = frame[c].astype(str).str.strip().str.lower().map(
            {"yes": 1, "true": 1, "1": 1, "no": 0, "false": 0, "0": 0}
        ).astype(float)
    out["income_freq"] = frame["Annual_Income_USD"].map(income_counts).fillna(0).astype(float)
    for c, (scale, width) in DIGIT_SPECS.items():
        scaled = (pd.to_numeric(frame[c], errors="coerce") * scale).round().astype("Int64")
        for pos in range(width):
            digit = ((scaled.abs() // 10**pos) % 10).astype("Int8")
            out[f"digit__{c}__{pos}"] = digit.astype(str).fillna("__MISSING__")
    return out


income_counts = pd.concat([train["Annual_Income_USD"], test["Annual_Income_USD"]]).value_counts()
X = build_features(train, income_counts)
X_test = build_features(test, income_counts)
CAT_COLS = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
print(X.shape, "categorical columns:", len(CAT_COLS))
X.head(3)


## Stratified 5-fold TabM

One `TabM_D_Classifier` per fold. With `FAST = True` (default here) the model uses k = 8 ensemble
members, batch size 2048, lr 5e-3, at most 30 epochs with patience 5: on a Kaggle T4 that is a few
minutes per fold instead of ~20. The pytabkit defaults (k = 32, batch 256, up to 200 epochs) are one
flag away and score about 0.0005 higher. The fold's validation split is used for early stopping,
and the validation metric is `1 - AUC` to match the competition metric.


In [ ]:
if SMOKE:
    rng = np.random.default_rng(SEED)
    keep = rng.choice(len(X), 20_000, replace=False)
    X, y = X.iloc[keep].reset_index(drop=True), y[keep]
    X_test = X_test.iloc[:2_000].reset_index(drop=True)

oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (fit_idx, val_idx) in enumerate(skf.split(X, y)):
    t0 = time.time()
    model = TabM_D_Classifier(
        device=DEVICE,
        random_state=SEED + fold,
        n_threads=4,
        verbosity=0,
        val_metric_name="1-auc_ovr",
        # Fast settings: ~8-12x quicker than the pytabkit defaults (k=32, batch 256, up to 200 epochs)
        # for roughly -0.0005 OOF AUC. Set FAST = False to use the defaults.
        tabm_k=8 if FAST else 32,
        batch_size=2048 if FAST else 256,
        lr=5e-3 if FAST else 2e-3,
        n_epochs=2 if SMOKE else (30 if FAST else 200),
        patience=5 if FAST else 16,
    )
    model.fit(
        X.iloc[fit_idx], y[fit_idx],
        X_val=X.iloc[val_idx], y_val=y[val_idx],
        cat_col_names=CAT_COLS,
    )
    oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"fold {fold}: AUC = {roc_auc_score(y[val_idx], oof[val_idx]):.6f}  ({time.time() - t0:.0f}s)")

print(f"OOF AUC = {roc_auc_score(y, oof):.6f}")


In [ ]:
submission = pd.DataFrame({ID: test[ID].iloc[: len(X_test)].to_numpy(), TARGET: test_pred})
submission.to_csv("submission.csv", index=False)
submission.head()


## Notes

- The local 5-fold OOF AUC of this recipe with the pytabkit defaults was about **0.9447**; the fast
  settings used here land around **0.944**. The strongest single
  gradient-boosting models sit around 0.9459, so TabM is not the best single model here.
- Its value shows up in ensembles: when I stacked ~200 diverse models with a nested logistic
  regression, the TabM variants without target encoding received the largest weights, because
  their errors are the least correlated with the tree models.
- Easy next steps: add `num_emb_type="pwl"` (piecewise-linear numeric embeddings), which gave
  +0.001 OOF AUC on a view with target encodings; average a few seeds; or blend with a CatBoost
  baseline on the logit scale.
